# ADHD Tools Colab Notebook

Run each cell to interact with simplified versions of the tools. This notebook stores data in a folder named **ADHDtools** inside your Google Drive.


In [ ]:
from google.colab import drive
import os, json

drive.mount('/content/drive')
DATA_DIR = '/content/drive/My Drive/ADHDtools'
os.makedirs(DATA_DIR, exist_ok=True)

def load_data(name, default=None):
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return default if default is not None else {}

def save_data(name, data):
    with open(os.path.join(DATA_DIR, name), 'w') as f:
        json.dump(data, f)

# Shared tasks list used by multiple tools
tasks_data = load_data('tasks.json', [])


## Pomodoro Timer

In [ ]:
import time, threading
import ipywidgets as widgets
from IPython.display import display, Audio

sequence_in = widgets.Text(value='25,5', description='Seq (min):')
start_btn = widgets.Button(description='Start')
stop_btn = widgets.Button(description='Stop')
progress = widgets.IntProgress(value=0)
status = widgets.Label(value='Ready')
display(sequence_in, start_btn, stop_btn, progress, status)

stop_flag = False

def run_pomodoro():
    global stop_flag
    try:
        seq = [int(x.strip()) for x in sequence_in.value.split(',') if x.strip()]
    except ValueError:
        status.value = 'Invalid sequence'
        return
    logs = load_data('pomodoro.json', {'logs': []})
    start_time = time.time()
    for idx, mins in enumerate(seq):
        if stop_flag:
            break
        phase = 'Work' if idx % 2 == 0 else 'Break'
        seconds = mins * 60
        progress.max = seconds
        progress.value = 0
        while seconds > 0 and not stop_flag:
            m, s = divmod(seconds, 60)
            status.value = f"{phase} {int(m):02d}:{int(s):02d}"
            progress.value = progress.max - seconds
            time.sleep(1)
            seconds -= 1
        if stop_flag:
            break
        display(Audio(url='https://actions.google.com/sounds/v1/alarms/beep_short.ogg', autoplay=True))
    status.value = 'Done!' if not stop_flag else 'Stopped'
    logs['logs'].append({'start': start_time, 'sequence': seq})
    save_data('pomodoro.json', logs)


def on_start(_):
    global stop_flag
    stop_flag = False
    threading.Thread(target=run_pomodoro).start()


def on_stop(_):
    global stop_flag
    stop_flag = True

start_btn.on_click(on_start)
stop_btn.on_click(on_stop)


## Eisenhower Matrix

In [ ]:
import ipywidgets as widgets
from IPython.display import display

categories = ['Do Now', 'Schedule', 'Delegate', 'Delete']
tasks = load_data('eisenhower.json', {c: [] for c in categories})

task_in = widgets.Text(description='Task:')
cat_dd = widgets.Dropdown(options=categories, description='Category:')
add_btn = widgets.Button(description='Add')
matrix_box = widgets.VBox()


def render_matrix():
    cols = []
    for c in categories:
        header = widgets.HTML(f"<b>{c}</b>")
        items = []
        for idx, t in enumerate(tasks[c]):
            lbl = widgets.Label(t)
            del_btn = widgets.Button(description='Delete', layout=widgets.Layout(width='60px'))
            def make_cb(cat=c, i=idx):
                def cb(_):
                    tasks[cat].pop(i)
                    save_data('eisenhower.json', tasks)
                    render_matrix()
                return cb
            del_btn.on_click(make_cb())
            items.append(widgets.HBox([lbl, del_btn]))
        cols.append(widgets.VBox([header]+items))
    matrix_box.children = cols


def add_task(_):
    t = task_in.value.strip()
    if t:
        tasks[cat_dd.value].append(t)
        save_data('eisenhower.json', tasks)
        if t not in [x[0] for x in tasks_data]:
            tasks_data.append([t, ''])
            save_data('tasks.json', tasks_data)
        task_in.value = ''
        render_matrix()

add_btn.on_click(add_task)

display(task_in, cat_dd, add_btn, matrix_box)
render_matrix()


## Day Planner

In [ ]:
import pandas as pd
from IPython.display import display

planner = load_data('planner.json', [])

existing_dd = widgets.Dropdown(options=[''] + [t[0] for t in tasks_data], description='From Tasks:')
task_in = widgets.Text(description='Task:')
time_in = widgets.Text(description='Time (HH:MM):')
add_btn = widgets.Button(description='Add')
table_box = widgets.VBox()

def render_plan():
    rows = []
    for idx, (tm, t) in enumerate(planner):
        lbl = widgets.Label(f"{tm} - {t}")
        del_btn = widgets.Button(description='Delete', layout=widgets.Layout(width='60px'))
        def make_cb(i=idx):
            def cb(_):
                planner.pop(i)
                save_data('planner.json', planner)
                render_plan()
            return cb
        del_btn.on_click(make_cb())
        rows.append(widgets.HBox([lbl, del_btn]))
    table_box.children = rows

def add_plan(_):
    t = existing_dd.value or task_in.value.strip()
    tm = time_in.value.strip()
    if t and tm:
        planner.append([tm, t])
        planner.sort()
        save_data('planner.json', planner)
        if t not in [x[0] for x in tasks_data]:
            tasks_data.append([t, ''])
            save_data('tasks.json', tasks_data)
        existing_dd.options = [''] + [x[0] for x in tasks_data]
        task_in.value=''
        time_in.value=''
        existing_dd.value=''
        render_plan()

add_btn.on_click(add_plan)

display(existing_dd, task_in, time_in, add_btn, table_box)
render_plan()


## Task Manager

In [ ]:
text_task = widgets.Text(description='Task:')
priority = widgets.Dropdown(options=['Low','Medium','High'], value='Medium', description='Priority:')
add_btn = widgets.Button(description='Add')
tasks_box = widgets.VBox()

def render_tasks():
    rows=[]
    for idx,(t,p) in enumerate(tasks_data):
        lbl=widgets.Label(f"{t} ({p})")
        del_btn=widgets.Button(description='Delete', layout=widgets.Layout(width='60px'))
        def make_cb(i=idx):
            def cb(_):
                tasks_data.pop(i)
                save_data('tasks.json', tasks_data)
                render_tasks()
            return cb
        del_btn.on_click(make_cb())
        rows.append(widgets.HBox([lbl, del_btn]))
    tasks_box.children=rows

def add_task(_):
    t=text_task.value.strip()
    if t:
        tasks_data.append([t, priority.value])
        save_data('tasks.json', tasks_data)
        text_task.value=''
        render_tasks()

add_btn.on_click(add_task)

display(text_task, priority, add_btn, tasks_box)
render_tasks()


## Task Breakdown

In [ ]:
breakdowns = load_data('breakdown.json', {})

main_task_in = widgets.Text(description='Main Task:')
add_main_btn = widgets.Button(description='Add Main Task')
main_select = widgets.Dropdown(options=list(breakdowns.keys()), description='Select:')
subtask_in = widgets.Text(description='Subtask:')
add_sub_btn = widgets.Button(description='Add Subtask')
output = widgets.VBox()


def render_breakdown():
    main_select.options = list(breakdowns.keys())
    rows = []
    for m, subs in breakdowns.items():
        del_main = widgets.Button(description='Del', layout=widgets.Layout(width='40px'))
        def make_del_main(task=m):
            def cb(_):
                breakdowns.pop(task, None)
                save_data('breakdown.json', breakdowns)
                render_breakdown()
            return cb
        del_main.on_click(make_del_main())
        header = widgets.HBox([widgets.HTML(f"<b>{m}</b>"), del_main])
        sub_items = []
        for idx, s in enumerate(subs):
            del_btn = widgets.Button(description='Del', layout=widgets.Layout(width='40px'))
            lbl = widgets.Label(s)
            def make_del_sub(ma=m, i=idx):
                def cb(_):
                    breakdowns[ma].pop(i)
                    save_data('breakdown.json', breakdowns)
                    render_breakdown()
                return cb
            del_btn.on_click(make_del_sub())
            sub_items.append(widgets.HBox([lbl, del_btn]))
        rows.append(widgets.VBox([header]+sub_items))
    output.children = rows


def add_main(_):
    m = main_task_in.value.strip()
    if m and m not in breakdowns:
        breakdowns[m] = []
        save_data('breakdown.json', breakdowns)
        if m not in [x[0] for x in tasks_data]:
            tasks_data.append([m,''])
            save_data('tasks.json', tasks_data)
        main_task_in.value = ''
        render_breakdown()


def add_sub(_):
    m = main_select.value
    s = subtask_in.value.strip()
    if m and s:
        breakdowns[m].append(s)
        save_data('breakdown.json', breakdowns)
        subtask_in.value = ''
        render_breakdown()

add_main_btn.on_click(add_main)
add_sub_btn.on_click(add_sub)

display(main_task_in, add_main_btn, main_select, subtask_in, add_sub_btn, output)
render_breakdown()


## Habit Tracker

In [ ]:
import datetime

habits = load_data('habits.json', {})

habit_in = widgets.Text(description='Habit:')
add_habit_btn = widgets.Button(description='Add Habit')
del_dd = widgets.Dropdown(options=list(habits.keys()), description='Delete:')
del_btn = widgets.Button(description='Delete')
tracker_out = widgets.Output()


def render_habits():
    del_dd.options = list(habits.keys())
    tracker_out.clear_output()
    with tracker_out:
        today = datetime.date.today()
        dates = [today + datetime.timedelta(days=i) for i in range(7)]
        header = ['Habit'] + [d.strftime('%m-%d') for d in dates]
        rows = []
        for h, checks in habits.items():
            row = [h]
            for d in dates:
                key = d.isoformat()
                row.append('✓' if checks.get(key) else '')
            rows.append(row)
        df = pd.DataFrame(rows, columns=header)
        display(df)


def add_habit(_):
    h = habit_in.value.strip()
    if h and h not in habits:
        habits[h] = {}
        save_data('habits.json', habits)
        habit_in.value = ''
        render_habits()

def delete_habit(_):
    h = del_dd.value
    if h in habits:
        habits.pop(h)
        save_data('habits.json', habits)
        render_habits()

add_habit_btn.on_click(add_habit)
del_btn.on_click(delete_habit)

display(habit_in, add_habit_btn, del_dd, del_btn, tracker_out)
render_habits()


## Routine Tool

In [ ]:
routine = load_data('routine.json', [])

name_in = widgets.Text(description='Step:')
duration_in = widgets.IntText(description='Minutes:', value=1)
add_step_btn = widgets.Button(description='Add Step')
start_btn = widgets.Button(description='Start Routine')
stop_btn = widgets.Button(description='Stop Routine')
steps_box = widgets.VBox()
output = widgets.Output()


def render_routine():
    rows=[]
    for idx, step in enumerate(routine):
        lbl=widgets.Label(f"{step['name']} - {step['minutes']} min")
        del_btn=widgets.Button(description='Delete', layout=widgets.Layout(width='60px'))
        def make_del(i=idx):
            def cb(_):
                routine.pop(i)
                save_data('routine.json', routine)
                render_routine()
            return cb
        del_btn.on_click(make_del())
        rows.append(widgets.HBox([lbl, del_btn]))
    steps_box.children=rows


def add_step(_):
    name=name_in.value.strip()
    mins=duration_in.value
    if name:
        routine.append({'name': name, 'minutes': mins})
        save_data('routine.json', routine)
        if name not in [x[0] for x in tasks_data]:
            tasks_data.append([name,''])
            save_data('tasks.json', tasks_data)
        name_in.value=''
        duration_in.value=1
        render_routine()

running = False

def run_routine():
    global running
    running = True
    for step in routine:
        if not running:
            break
        output.append_stdout('\n'+step['name']+'\n')
        for i in range(step['minutes'] * 60, 0, -1):
            if not running:
                break
            output.append_stdout(f"\r{i//60:02d}:{i%60:02d}")
            time.sleep(1)
        output.append_stdout('\n')
    running = False

def start_routine_btn(_):
    if not running:
        threading.Thread(target=run_routine).start()

def stop_routine_btn(_):
    global running
    running = False

add_step_btn.on_click(add_step)
start_btn.on_click(start_routine_btn)
stop_btn.on_click(stop_routine_btn)

display(name_in, duration_in, add_step_btn, steps_box, start_btn, stop_btn, output)
render_routine()


## Rewards

In [ ]:
rewards = load_data('rewards.json', {'points': 0})
points_label = widgets.Label()
inc_btn = widgets.Button(description='Add Point')


def render_points():
    points_label.value = f"Points: {rewards['points']}"


def add_point(b):
    rewards['points'] += 1
    save_data('rewards.json', rewards)
    render_points()

inc_btn.on_click(add_point)

display(inc_btn, points_label)
render_points()


## Calendar Tool

In [ ]:
!pip -q install ics
from ics import Calendar

ics_path = widgets.Text(description='ICS Path:')
load_btn = widgets.Button(description='Load')
cal_out = widgets.Output()


def load_ics(b):
    path = ics_path.value
    if os.path.exists(path):
        with open(path, 'r') as f:
            c = Calendar(f.read())
        cal_out.clear_output()
        with cal_out:
            for e in c.events:
                print(e.begin.humanize(), '-', e.name)

load_btn.on_click(load_ics)

display(ics_path, load_btn, cal_out)


All tools above save data to your Google Drive folder so you can access them later on any device. Enjoy!